# B5a：歷史 PIT Universe 對帳（離線，不需網路）

用已封存 832 個交易日的官方日行情當每日名單，對照 FinLab 價格、富邦 5 分 K（gap ledger）、官方上下市、暫停交易、轉板。
上市、可交易、暫停、有價格、官方下市分開記錄；缺漏逐筆分類；最後價格日與官方下市日分開；不在最後價格自動平倉。
輸出 `universe_panel.parquet`、`universe_symbols.csv`、`coverage_breakdown.csv`、`board_transfers.csv`、各類缺漏 csv、`manifest.json`。

**依序執行全部格子即可**，約 5–15 分鐘。

In [ ]:
# 1. 參數
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
LAKE = f"{DATA_ROOT}/_tw-market-data-lake"
CA_ROOT = f"{LAKE}/reference/corporate_actions/v1"
OFFICIAL_ROOT = f"{LAKE}/reference/official_volume/v1"
GAP_LEDGER = f"{LAKE}/audits/gap_ledger_20261026_163358_3e8eed0/gap_day_summary.parquet"
PROJECT_DIR = f"/content/{REPO_NAME}"
OUT_DIR = "/content/tw_lake_universe_pit"
ARCHIVE = f"{LAKE}/audits/universe_pit"
PREFIX = "universe_pit"


In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3. clone / pull + 依賴
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def ensure_drive():
    """Mount Drive safely. If an earlier cell unmounted Drive and then wrote under /content/drive,
    those files are only on this runtime's disk: move them aside (never delete) and copy any audit
    folders that are missing on the real Drive."""
    import os, shutil, time
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        return
    stray = None
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        stray = f"/content/drive_local_{int(time.time())}"
        shutil.move("/content/drive", stray)
        print("moved local (unsynced) files to", stray)
    drive.mount("/content/drive")
    if stray:
        src_root = f"{stray}/MyDrive/00Quant_Research/data/_tw-market-data-lake/audits"
        dst_root = f"{DATA_ROOT}/_tw-market-data-lake/audits"
        for dirpath, dirnames, filenames in os.walk(src_root):
            rel = os.path.relpath(dirpath, src_root)
            for d in list(dirnames):
                target = os.path.join(dst_root, rel, d)
                if not os.path.exists(target):
                    shutil.copytree(os.path.join(dirpath, d), target)
                    print("copied missing", target)
                    dirnames.remove(d)

In [ ]:
# 4. 執行
print("exit", stream([sys.executable, "-m", "tw_lake.audit.universe_pit", "--official-root", OFFICIAL_ROOT,
                      "--ca-root", CA_ROOT, "--data-root", DATA_ROOT, "--gap-ledger", GAP_LEDGER, "--out-root", OUT_DIR]))


In [ ]:
# 5. 重點結果，並複製到 Drive（把這格輸出貼給 Claude）
import glob, json, os, shutil
import pandas as pd
ensure_drive()
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 300)
d = sorted(glob.glob(f"{OUT_DIR}/{PREFIX}_*"))[-1]
s = json.load(open(f"{d}/summary.json", encoding="utf-8"))
print(json.dumps({k: v for k, v in s.items() if k != "coverage_class_by_type"}, ensure_ascii=False, indent=1, default=str))
print("\n== coverage class by type"); print(pd.DataFrame(s["coverage_class_by_type"]).pivot_table(index="security_type", columns="coverage_class", values="n", fill_value=0).to_string())
cov = pd.read_csv(f"{d}/coverage_breakdown.csv")
for k in (["year", "exchange"], ["security_type"], ["liquidity_bucket"], ["termination_status"]):
    g = cov.groupby(k)[["official_traded_days", "finlab_priced", "fubon_bars"]].sum()
    g["missing_rate"] = 1 - g["finlab_priced"] / g["official_traded_days"]
    print("\n== coverage by", k); print(g.to_string())
sym = pd.read_csv(f"{d}/universe_symbols.csv", dtype={"symbol": str})
print("\n== delisted / stopped symbols"); print(sym[sym["termination_status"] != "ACTIVE"][["symbol", "security_type", "exchanges", "last_official_date", "last_observed_price_date", "official_delisting_date", "termination_status", "price_missing_traded_days", "price_after_delisting_days"]].to_string(index=False))
print("\n== board transfers"); print(pd.read_csv(f"{d}/board_transfers.csv", dtype={"symbol": str}).to_string(index=False))
m = pd.read_csv(f"{d}/official_traded_price_missing.csv", dtype={"symbol": str})
print("\n== official traded, FinLab price missing:", len(m)); print(m.groupby(["security_type", "exchange"]).size().to_string() if len(m) else "")
print(m.groupby("symbol").size().sort_values(ascending=False).head(30).to_string() if len(m) else "")
mc = pd.read_csv(f"{d}/official_traded_price_missing_causes.csv", dtype={"symbol": str})
print("\n== price-missing causes by type x exchange"); print(mc.groupby(["security_type", "exchange", "likely_cause"]).size().to_string() if len(mc) else "")
print("\n== unexplained missing, top symbols"); print(mc[mc["likely_cause"] == "UNEXPLAINED"].groupby("symbol").size().sort_values(ascending=False).head(30).to_string() if len(mc) else "")
print("\n== stale delisting records ignored"); print(pd.read_csv(f"{d}/stale_delisting_records.csv", dtype={"symbol": str}).to_string(index=False))
os.makedirs(ARCHIVE, exist_ok=True)
dst = f"{ARCHIVE}/{os.path.basename(d)}"
if not os.path.exists(dst):
    shutil.copytree(d, dst)
print("\ncopied to", dst)
